# 简单扩散模型实现

在这个笔记本中，我们将实现一个简单的一维扩散模型。我们的目标是：
1. 实现完整的前向扩散过程
2. 实现一个简单的U-Net模型作为噪声预测器
3. 实现反向扩散过程（采样）
4. 训练模型生成简单的一维信号

这个实现将帮助我们理解项目中更复杂的轨迹生成模型的工作原理。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

class SimpleDiffusion:
    def __init__(self, num_timesteps=1000):
        self.num_timesteps = num_timesteps
        # 线性beta schedule
        self.beta = torch.linspace(1e-4, 0.02, num_timesteps)
        self.alpha = 1. - self.beta
        self.alpha_bar = torch.cumprod(self.alpha, dim=0)
        
    def q_sample(self, x_0, t, noise=None):
        """前向过程：添加噪声"""
        if noise is None:
            noise = torch.randn_like(x_0)
            
        sqrt_alpha_bar = torch.sqrt(self.alpha_bar[t])
        sqrt_one_minus_alpha_bar = torch.sqrt(1 - self.alpha_bar[t])
        
        return sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * noise
    
    def p_sample(self, model, x_t, t):
        """反向过程：单步去噪"""
        with torch.no_grad():
            predicted_noise = model(x_t, t)
            
        alpha = self.alpha[t]
        alpha_bar = self.alpha_bar[t]
        beta = self.beta[t]
        
        x_0_predicted = (1 / torch.sqrt(alpha)) * (x_t - (beta / torch.sqrt(1 - alpha_bar)) * predicted_noise)
        if t > 0:
            noise = torch.randn_like(x_t)
            x_t_prev = x_0_predicted + torch.sqrt(beta) * noise
        else:
            x_t_prev = x_0_predicted
            
        return x_t_prev


In [ ]:
class SimpleUNet(nn.Module):
    """简单的一维U-Net模型，用于预测噪声"""
    def __init__(self, input_dim=100, time_dim=32):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim)
        )
        
        # 编码器
        self.enc1 = nn.Conv1d(1, 32, 3, padding=1)
        self.enc2 = nn.Conv1d(32, 64, 3, padding=1)
        self.enc3 = nn.Conv1d(64, 128, 3, padding=1)
        
        # 解码器
        self.dec3 = nn.Conv1d(128 + 64, 64, 3, padding=1)
        self.dec2 = nn.Conv1d(64 + 32, 32, 3, padding=1)
        self.dec1 = nn.Conv1d(32, 1, 3, padding=1)
        
        self.pool = nn.MaxPool1d(2)
        self.upsample = nn.Upsample(scale_factor=2)
        
    def forward(self, x, t):
        # 时间编码
        t = t.float().view(-1, 1)
        t = self.time_mlp(t)
        
        # 将输入reshape为(batch, channel, length)
        x = x.view(-1, 1, x.shape[-1])
        
        # 编码
        x1 = F.relu(self.enc1(x))
        x2 = F.relu(self.enc2(self.pool(x1)))
        x3 = F.relu(self.enc3(self.pool(x2)))
        
        # 解码
        x = F.relu(self.dec3(torch.cat([self.upsample(x3), x2], dim=1)))
        x = F.relu(self.dec2(torch.cat([self.upsample(x), x1], dim=1)))
        x = self.dec1(x)
        
        return x.view(x.shape[0], -1)


In [ ]:
# 创建训练数据：简单的正弦波
def create_data(num_samples=1000, seq_length=100):
    t = torch.linspace(0, 2*np.pi, seq_length)
    data = []
    for _ in range(num_samples):
        # 随机相位和振幅的正弦波
        phase = torch.rand(1) * 2 * np.pi
        amplitude = 0.5 + torch.rand(1)
        signal = amplitude * torch.sin(t + phase)
        data.append(signal)
    return torch.stack(data)

# 创建训练数据
train_data = create_data()

# 创建模型和扩散过程
model = SimpleUNet()
diffusion = SimpleDiffusion(num_timesteps=100)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 可视化一些训练数据
plt.figure(figsize=(10, 4))
for i in range(5):
    plt.plot(train_data[i].numpy())
plt.title('Training Data Examples')
plt.show()


In [ ]:
# 训练循环
num_epochs = 100
batch_size = 32

for epoch in range(num_epochs):
    total_loss = 0
    for i in range(0, len(train_data), batch_size):
        batch = train_data[i:i+batch_size]
        optimizer.zero_grad()
        
        # 随机时间步
        t = torch.randint(0, diffusion.num_timesteps, (len(batch),))
        # 添加噪声
        noise = torch.randn_like(batch)
        x_t = diffusion.q_sample(batch, t, noise)
        # 预测噪声
        predicted_noise = model(x_t, t)
        
        # 计算损失
        loss = F.mse_loss(predicted_noise, noise)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}, Average Loss: {total_loss / (len(train_data) / batch_size):.4f}')


In [ ]:
# 生成新的样本
@torch.no_grad()
def sample(model, diffusion, num_samples=1):
    # 从纯噪声开始
    x = torch.randn(num_samples, 100)
    
    # 逐步去噪
    for t in tqdm(range(diffusion.num_timesteps-1, -1, -1)):
        t_batch = torch.full((num_samples,), t, dtype=torch.long)
        x = diffusion.p_sample(model, x, t_batch)
    
    return x

# 生成并可视化样本
samples = sample(model, diffusion, num_samples=5)

plt.figure(figsize=(10, 4))
for i in range(len(samples)):
    plt.plot(samples[i].numpy())
plt.title('Generated Samples')
plt.show()
